# GPT From Scratch — Exploration
Sprint Day 9 | 2026-04-30

In [ ]:
# ── Setup (run once) ──────────────────────────────────────────────────────────
# Clone repo (Colab only — skip if running locally)
import subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['git', 'clone', 'https://github.com/sa1-kumar/gpt-from-scratch.git'])
    %cd gpt-from-scratch

# Install dependencies
!pip install torch numpy matplotlib wandb --quiet

import sys
sys.path.append('src')

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from attention import SingleHeadAttention, MultiHeadAttention, TransformerBlock
from gpt import GPT

print(f'PyTorch: {torch.__version__}')
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Single-Head Attention

In [ ]:
B, T, d_model, d_k = 2, 4, 512, 64
x = torch.randn(B, T, d_model)
sha = SingleHeadAttention(d_model, d_k)
out, weights = sha(x, causal=True)
print(f'Output:  {out.shape}')     # [2, 4, 64]
print(f'Weights: {weights.shape}') # [2, 4, 4]

# Visualize causal mask
plt.figure(figsize=(4, 4))
plt.imshow(weights[0].detach(), cmap='Blues')
plt.title('Single-Head Attention Weights (causal)')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.colorbar()
plt.show()

## 2. Multi-Head Attention

In [ ]:
B, T, d_model, n_heads = 2, 4, 512, 8
x = torch.randn(B, T, d_model)
mha = MultiHeadAttention(d_model, n_heads)
out, weights = mha(x, causal=True)
print(f'Output:  {out.shape}')     # [2, 4, 512]
print(f'Weights: {weights.shape}') # [2, 8, 4, 4]

# Visualize all 8 heads
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(weights[0, i].detach(), cmap='Blues')
    ax.set_title(f'Head {i+1}')
    ax.axis('off')
plt.suptitle('Multi-Head Attention Weights (causal)')
plt.tight_layout()
plt.show()

## 3. Transformer Block

In [ ]:
block = TransformerBlock(d_model=512, n_heads=8)
out, weights = block(x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')                      # [2, 4, 512]
print(f'Shape preserved: {x.shape == out.shape}')  # True

## 4. Full GPT — Sanity Check

In [ ]:
model = GPT(
    vocab_size=65,
    d_model=128,
    n_heads=4,
    n_layers=3,
    max_seq_len=64,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

idx     = torch.randint(0, 65, (2, 32)).to(device)
targets = torch.randint(0, 65, (2, 32)).to(device)
logits, loss = model(idx, targets)

print(f'Logits: {logits.shape}')       # [2, 32, 65]
print(f'Loss:   {loss.item():.4f}')    # ~4.17 = ln(65)
print(f'Expected loss (random init): {torch.log(torch.tensor(65.0)):.4f}')

## 5. Generation Test

In [ ]:
prompt    = torch.zeros((1, 1), dtype=torch.long).to(device)
generated = model.generate(prompt, max_new_tokens=50, temperature=0.8, top_k=10)
print(f'Generated shape: {generated.shape}')  # [1, 51]
print(f'Token ids: {generated[0].tolist()}')